# What a Probabilistic Forecast Is Actually Worth

## Objectives

Chapter 2 found a real, checked way to share information across Chapter 4's
customer archetypes without training one model per customer. Every result so
far has been a point forecast, though, a single number with no attached
sense of how wrong it might be. A real deployment decision often depends on
that missing piece: a risk-of-exceedance question needs a full density, a
transformer-sizing question needs a specific quantile, and a compliance
statement needs a guarantee that holds regardless of whether the model
itself is well-specified. This chapter builds one real backbone, MLPGAM,
into three different probabilistic paradigms, checks what each is actually
worth, and tests directly whether Chapter 2's own archetypes help here too,
or whether that idea, real and checked in Chapter 2, simply does not
transfer to this different question.

By the end you will be able to:

- Train a real parametric distributional forecast (`MLPGAMNormal`) and a
  real quantile forecast (`MLPGAMFpqr`) on the same 342-customer AusNet pool.
- Wrap either backbone in a genuine split-conformal calibration, the same
  finite-sample coverage guarantee Chapter 2 already used, and check whether
  an uncalibrated model's own stated uncertainty can be trusted as-is.
- Check honestly whether calibrating per Chapter 2's own archetypes sharpens
  a conformal interval, or whether it does not, and explain the real
  mechanism behind whichever answer the data gives.

## Getting the data, and one backbone, three paradigms

Same real AusNet pool the whole of Part 5 has used, same last-90-day
holdout. Every paradigm below shares one backbone architecture, MLPGAM, so
differences in the results come from the probabilistic paradigm itself, not
from comparing unrelated architectures.

- **Parametric**: `MLPGAMNormal` predicts a full Normal density,
  $\hat{y}_{i,t+h} \sim \mathcal{N}(\mu_{i,t+h}, \sigma_{i,t+h}^2)$, the only
  paradigm here that directly answers "what is the probability this
  customer's load exceeds $X$ kW."
- **Quantile**: `MLPGAMFpqr` predicts several quantile levels
  $\hat{q}_{i,t+h}^{(\tau)}$ directly, no distributional assumption, a direct
  answer to a capacity question like "what P95 load must a transformer be
  sized to survive."
- **Conformal**: a split-conformal wrapper around the parametric backbone's
  own $(\mu, \sigma)$, the only one of the three with an actual finite-sample
  coverage guarantee, reusing the exact calibration Chapter 2 already built,
  not a new mechanism.

<div class='ark-concept'>

<span class='ark-concept-title'><i class="bi bi-info-circle-fill"></i> Key Concept</span>

A point forecast and a probabilistic forecast answer different questions,
and the three probabilistic paradigms here answer different questions from
each other too. A parametric density is the only one that can price a
risk-of-exceedance question directly. A quantile forecast is the only one
with no distributional assumption baked in. And conformal calibration is
the only one with an actual guarantee: under exchangeability, at least
$1-\alpha$ of a calibration population's own true outcomes fall inside the
interval by construction [@lei2018distributionfree], regardless of whether
the underlying model's own distributional assumptions are correct. None of
the three is strictly better; each is the right tool for a different real
decision.

</div>

![Three paradigms, one backbone. A parametric density answers a risk-of-exceedance question directly. A quantile forecast answers a capacity question with no distributional assumption. Conformal calibration wraps either one in a finite-sample coverage guarantee that holds regardless of whether the backbone's own assumptions are correct.](../../assets/three-uq-paradigms.svg){#fig-three-uq-paradigms}

In [ ]:
import contextlib
import io
import logging
import warnings

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)  # Lightning's own GPU/TPU/tip/deprecation noise, not this notebook's own output


@contextlib.contextmanager
def quiet_training():
    """Suppress Lightning's own model-summary table and progress-bar widget.

    Both render via IPython's rich display() protocol inside a Jupyter kernel,
    not stdout, so a stdout redirect alone does not catch them; silencing
    display() itself for the duration of one fit() call does, verified directly.
    """
    import IPython.display as ipy_display

    original_display = ipy_display.display
    ipy_display.display = lambda *a, **k: None
    try:
        yield
    finally:
        ipy_display.display = original_display


from pathlib import Path

from great_tables import GT, md
from lets_plot import (
    LetsPlot,
    aes,
    element_text,
    geom_bar,
    ggplot,
    ggsize,
    labs,
    scale_fill_manual,
    theme,
)
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
import torch

from ark.plot.gt_style import themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import DANGER, PRIMARY, SUCCESS, WARNING

LetsPlot.setup_html(isolated_frame=True)

UNBOLD_AXES = theme(axis_title_x=element_text(face="plain"), axis_title_y=element_text(face="plain"))
FULL_WIDTH = 960
STEPS_PER_DAY = 48
LOOKBACK = STEPS_PER_DAY  # one real day of context
HORIZON = STEPS_PER_DAY  # 24-hour-ahead, same horizon as Chapters 1-2
TEST_STEPS = 90 * STEPS_PER_DAY  # the same last-90-day holdout
N_CLUSTERS = 5
DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")
RNG = np.random.default_rng(42)

# One fixed color per paradigm, reused in every chart in this notebook.
PARADIGM_COLORS = {
    "Parametric (Normal)": PRIMARY,
    "Quantile (FPQR)": WARNING,
    "Conformal, global": SUCCESS,
    "Conformal, per-archetype": DANGER,
}


def normalize_shape(profiles: np.ndarray) -> np.ndarray:
    peak = profiles.max(axis=-1, keepdims=True)
    peak = np.where(peak == 0, 1, peak)
    return profiles / peak


load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
n_customers = load_data.shape[0]
print(f"load_data: {load_data.shape} (customers, days, half-hours)")

# Chapter 2's own archetype recipe, reused verbatim: KMeans(k=5) on each
# customer's peak-normalized 90-day seasonal shape.
season = load_data[:, 0:90, :]
X_shape = normalize_shape(season.mean(axis=1))
km = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
archetype_labels = km.fit_predict(X_shape)

load_data: (342, 365, 48) (customers, days, half-hours)


Chapters 1 and 2 both fed a model a small, hand-picked table of features: the
most recent reading, the same half-hour a day ago, two days ago, a week ago,
plus the half-hour and the day of week. That recipe works well for a tree
model like LightGBM, which reasons about a fixed set of columns one split at
a time. MLPGAM reasons differently: it is a sequence model, and it expects a
real, continuous stretch of a customer's own history as its input, not a
curated handful of lag columns. Each training window here is one full real
day of lookback immediately followed by the real day the model has to
forecast, and the network is left to learn whatever temporal structure,
daily rhythm, the slow decay of yesterday's influence on today, actually
matters, directly from that raw sequence, rather than being told in advance
which four or five moments in the past are worth looking at.

In [ ]:
def build_windows(series: np.ndarray, *, lookback: int, horizon: int) -> tuple[np.ndarray, np.ndarray]:
    """Non-overlapping (X, y) windows: X is lookback+horizon combined, y is horizon only.

    Non-overlapping keeps every real day's forecast independent of its
    neighbors, matching the same one-window-per-real-day protocol Chapter 1's
    own trace charts used.
    """
    combined = lookback + horizon
    n_windows = (len(series) - combined) // horizon + 1
    x = np.zeros((n_windows, combined, 1), dtype=np.float32)
    y = np.zeros((n_windows, horizon, 1), dtype=np.float32)
    for i in range(n_windows):
        start = i * horizon
        x[i, :, 0] = series[start : start + combined]
        y[i, :, 0] = series[start + lookback : start + combined]
    return x, y


all_x, all_y, all_archetype, all_is_test = [], [], [], []
for cust_id in range(n_customers):
    x, y = build_windows(load_data[cust_id].reshape(-1), lookback=LOOKBACK, horizon=HORIZON)
    n = len(x)
    test_windows = TEST_STEPS // HORIZON
    is_test = np.arange(n) >= (n - test_windows)
    all_x.append(x)
    all_y.append(y)
    all_archetype.append(np.full(n, archetype_labels[cust_id]))
    all_is_test.append(is_test)

X_all = np.concatenate(all_x, axis=0)
Y_all = np.concatenate(all_y, axis=0)
archetype_all = np.concatenate(all_archetype, axis=0)
is_test_all = np.concatenate(all_is_test, axis=0)

X_train, Y_train = X_all[~is_test_all], Y_all[~is_test_all]
X_test, Y_test = X_all[is_test_all], Y_all[is_test_all]
archetype_test = archetype_all[is_test_all]
print(f"pooled windows: {len(X_all):,}, train: {len(X_train):,}, test: {len(X_test):,}")

pooled windows: 124,488, train: 93,708, test: 30,780


## The parametric density

The first paradigm asks MLPGAM to output two numbers for every half-hour of
the forecast horizon, not one: a mean $\mu_{i,t+h}$ and a scale
$\sigma_{i,t+h}$, together describing a full Normal density
$\mathcal{N}(\mu_{i,t+h}, \sigma_{i,t+h}^2)$ over that customer's own load at
that moment. Training looks different too: rather than minimizing a
pointwise error like MAE or MSE, the network is trained to maximize the
Normal log-likelihood of the real observed load under its own predicted
density, so it is rewarded directly for getting both the center and the
spread of its own uncertainty right, not just the center. That extra output
is not free, and it is not free for a good reason: a real operational
question like "what is the probability this customer's load exceeds 3 kW
tomorrow evening" has a genuine, closed-form answer once $\mu$ and $\sigma$
are known, something a single point forecast can never provide on its own,
however accurate that point happens to be.

In [ ]:
from twiga.core.config import DataPipelineConfig
from twiga.forecaster import get_model

data_config = DataPipelineConfig(
    target_feature="value", period="30min", forecast_horizon=HORIZON, lookback_window_size=LOOKBACK
)

normal_cls, normal_cfg_cls = get_model("mlpgamnormal")
normal_cfg = normal_cfg_cls.from_data_config(data_config)
normal_cfg.max_epochs = 15
normal_cfg.num_workers = 0
normal_cfg.rich_progress_bar = False
normal_model = normal_cls(normal_cfg)
# TwigaForecaster normally assigns this; set directly since this notebook
# pools all 342 customers into one training call rather than going through
# TwigaForecaster's own single-series data pipeline.
normal_model.data_pipeline = None
with quiet_training(), contextlib.redirect_stdout(io.StringIO()):
    normal_model.fit(X_train, Y_train)

normal_out = normal_model.forecast(torch.tensor(X_test))
loc, scale = normal_out["loc"].squeeze(-1), normal_out["scale"].squeeze(-1)
y_test_flat = Y_test.squeeze(-1)
normal_mae = np.abs(y_test_flat - loc).mean()
print(f"MLPGAMNormal test MAE: {normal_mae:.4f}")

Training: |          | 0/? [00:00<?, ?it/s]

MLPGAMNormal test MAE: 0.3627


## The quantile forecast

The Normal head above buys a full density, but it buys it by assuming the
residuals actually follow a Normal shape, symmetric, thin-tailed, fully
described by just a mean and a spread. Real household load does not always
oblige that assumption: evening peaks can be sharper on one side than the
other, and a handful of unusually large readings can sit well outside
anything a Normal curve expects. `MLPGAMFpqr`, the FPSeq2Q architecture,
sidesteps the assumption entirely. Rather than fitting one distributional
family, it is trained with a pinball loss across several quantile levels at
once, each level penalized separately for over- or under-shooting the real
outcome, so the model learns the P5, the P50, the P95, and everything
between directly from the data's own real shape, whatever that shape turns
out to be. The practical payoff is a direct answer to a capacity question a
{{< acr DSO >}} actually asks, "what P95 load must a transformer be sized to
survive," without ever assuming the underlying noise looks Gaussian.

In [ ]:
fpqr_cls, fpqr_cfg_cls = get_model("mlpgamfpqr")
fpqr_cfg = fpqr_cfg_cls.from_data_config(data_config)
fpqr_cfg.max_epochs = 15
fpqr_cfg.num_workers = 0
fpqr_cfg.rich_progress_bar = False
fpqr_model = fpqr_cls(fpqr_cfg)
fpqr_model.data_pipeline = None
with quiet_training(), contextlib.redirect_stdout(io.StringIO()):
    fpqr_model.fit(X_train, Y_train)

fpqr_out = fpqr_model.forecast(torch.tensor(X_test))
quantiles = fpqr_out["quantiles"].squeeze(-1)  # (B, n_quantiles, horizon)
q_levels = fpqr_out["quantile_levels"][0, :, 0]
p5_idx = int(np.argmin(np.abs(q_levels - 0.05)))
p95_idx = int(np.argmin(np.abs(q_levels - 0.95)))
fpqr_p5, fpqr_p95 = quantiles[:, p5_idx, :], quantiles[:, p95_idx, :]
fpqr_picp = ((y_test_flat >= fpqr_p5) & (y_test_flat <= fpqr_p95)).mean()
fpqr_width = (fpqr_p95 - fpqr_p5).mean()
print(f"FPQR raw P5-P95 empirical coverage: {fpqr_picp:.3f} (nominal target: 0.90)")
print(f"FPQR raw P5-P95 mean width: {fpqr_width:.4f}")

Training: |          | 0/? [00:00<?, ?it/s]

FPQR raw P5-P95 empirical coverage: 0.990 (nominal target: 0.90)
FPQR raw P5-P95 mean width: 3.0809


Look at the real gap between what FPQR's own P5-P95 band claims and what it
actually delivers: close to 99% empirical coverage against a nominal 90%
target. Nearly one in ten "misses" the model itself was designed to allow
for essentially never happens; the interval is far wider than it needs to
be, at real operational cost, since a transformer sized to a needlessly
inflated P95 is a transformer over-built for the actual risk. A raw,
uncalibrated model's own stated uncertainty, in other words, does not
automatically mean what it claims to mean, in either the parametric or the
quantile paradigm, and that real gap between the claimed and the actual is
exactly the problem split-conformal calibration exists to close, next.

## Split-conformal calibration: does it need calibrating at all?

FPQR's own over-coverage above raises an honest question about the
parametric head too: MLPGAMNormal already outputs a full $(\mu, \sigma)$ for
every forecast, so why not just trust $\sigma$ as a calibrated interval on
its own, the way the raw FPQR band was trusted a moment ago, and just proved
untrustworthy? The answer is to stop trusting either model's own stated
uncertainty at face value and calibrate it instead, using the exact same
split-conformal machinery Chapter 2 already built for the archetype
confidence gate, reused verbatim here, not reimplemented for a second time.
The idea is to hold back a calibration slice of real customers, measure how
far their true outcome actually sat from the model's own prediction relative
to its own claimed scale, and use the resulting empirical distribution to
correct the interval directly:
$$\tau = \text{Quantile}_{1-\alpha}\Big(\Big\{\, \frac{|y_j - \mu_j|}{\sigma_j} : j \in \text{calibration set} \,\Big\}\Big), \qquad [\mu_i - \tau\sigma_i,\ \mu_i + \tau\sigma_i]$$
This is a scaled non-conformity score, dividing each calibration residual by
that same customer's own predicted $\sigma$ rather than using an unscaled
residual directly, so a customer the model is already unsure about does not
get penalized twice. The resulting guarantee is a real, finite-sample one,
not asymptotic and not dependent on the Normal assumption being correct:
Lei, G'Sell, Rinaldo, Tibshirani and Wasserman formalized exactly this result
in 2018 [@lei2018distributionfree], under exchangeability, at least
$1-\alpha$ of a calibration population's own true outcomes fall inside the
resulting interval by construction, regardless of whether the underlying
model's own distributional assumptions happen to be right.

In [ ]:
from twiga.distributions.conformal.crc import ConformalResidualFitting

calib_idx, eval_idx = train_test_split(np.arange(len(X_test)), test_size=0.5, random_state=0)


def coverage_and_width(lower: np.ndarray, upper: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    covered = (y >= lower) & (y <= upper)
    return float(covered.mean()), float((upper - lower).mean())


cp_global = ConformalResidualFitting(alpha=0.1)
cp_global.calibrate(loc[calib_idx], scale[calib_idx], y_test_flat[calib_idx])
lower_g, upper_g = cp_global.generate_intervals(loc[eval_idx], scale[eval_idx])
cov_g, width_g = coverage_and_width(lower_g, upper_g, y_test_flat[eval_idx])
print(f"Global conformal: coverage={cov_g:.3f}, mean width={width_g:.4f}")

Global conformal: coverage=0.899, mean width=1.0317


## Does calibrating per Chapter 2's own archetypes sharpen this?

A single global $\tau$ treats every one of the 342 real customers as
interchangeable for calibration purposes, which is a strange thing to do in
a book that spent an entire chapter arguing customers are not
interchangeable. Chapter 2's own real archetypes are already sitting right
there, already computed, already validated on this same population, so the
natural next question is whether calibrating separately per archetype,
rather than once globally, produces a sharper interval at the same 90%
guarantee. This is not a new idea invented for this chapter: the author's
own exploratory notebook tried exactly this, calibrating per cluster instead
of using one global threshold, though it built a fresh {{< acr UMAP >}} and
{{< acr GMM >}} clustering pass just to run that check. Reusing Chapter 2's
own already-computed archetypes here instead avoids that duplicated
clustering effort entirely, and the real question is checked honestly below
rather than assumed to come out sharper just because conditioning on more
information usually sounds like it should help.

In [ ]:
lower_pa = np.zeros_like(loc[eval_idx])
upper_pa = np.zeros_like(loc[eval_idx])
tau_by_archetype = {}
for k in range(N_CLUSTERS):
    calib_k = calib_idx[archetype_test[calib_idx] == k]
    eval_k_mask = archetype_test[eval_idx] == k
    if len(calib_k) < 5 or eval_k_mask.sum() == 0:
        continue
    cp_k = ConformalResidualFitting(alpha=0.1)
    scores_k = cp_k.get_scores(loc[calib_k], scale[calib_k], y_test_flat[calib_k])
    tau_by_archetype[k] = float(np.mean(cp_k.calculate_conformal_quantile(scores_k, alpha=0.1)))
    cp_k.calibrate(loc[calib_k], scale[calib_k], y_test_flat[calib_k])
    lo_k, up_k = cp_k.generate_intervals(loc[eval_idx][eval_k_mask], scale[eval_idx][eval_k_mask])
    lower_pa[eval_k_mask], upper_pa[eval_k_mask] = lo_k, up_k

cov_pa, width_pa = coverage_and_width(lower_pa, upper_pa, y_test_flat[eval_idx])
print(f"Per-archetype conformal: coverage={cov_pa:.3f}, mean width={width_pa:.4f}")

comparison = pd.DataFrame(
    [
        {"Approach": "Global conformal", "Coverage": cov_g, "Mean width": width_g},
        {"Approach": "Per-archetype conformal", "Coverage": cov_pa, "Mean width": width_pa},
    ]
)
themed_gt(
    GT(comparison.round(4))
    .tab_header(title=md("**Per-Archetype Calibration Is Wider, Not Sharper**"))
    .tab_source_note("342 real AusNet customers, split-conformal on MLPGAMNormal's own (mu, sigma)"),
    n_rows=len(comparison),
)

Per-archetype conformal: coverage=0.899, mean width=1.1079


GT(_tbl_data=                  Approach  Coverage  Mean width
0         Global conformal    0.8994      1.0317
1  Per-archetype conformal    0.8988      1.1079, _body=<great_tables._gt_data.Body object at 0x14e766540>, _boxhead=Boxhead([ColInfo(var='Approach', type=<ColInfoTypeEnum.default: 1>, column_label='Approach', column_align='left', column_width=None), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='right', column_width=None), ColInfo(var='Mean width', type=<ColInfoTypeEnum.default: 1>, column_label='Mean width', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x14e7ab860>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Per-Archetype Calibration Is Wider, Not Sharper**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12f690a70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12f690ad0>, _source_notes=["342 real AusNet customers, split-conformal on MLPGAMNormal's own (mu, sigma)"], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Approach', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Coverage', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Mean width', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Approach', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Coverage', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Mean width', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')])], _locale=<great_tables._gt_data.Locale object at 0x12f690a40>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_ad

Both approaches hit the 90% coverage target they were calibrated for, so
neither one is broken in the sense of failing its own guarantee. But
per-archetype calibration is measurably wider than the single global
threshold, the opposite of what Chapter 2's own archetype-conditioning idea
delivered when the question was point-forecast accuracy rather than
interval calibration. That is a real, checked result, not the expected one,
and it is worth understanding exactly why rather than filing it away as a
curiosity. Two real diagnostics, computed directly on this same model and
this same calibration split, explain the mechanism.

In [ ]:
diagnostic_rows = []
for k, tau_k in tau_by_archetype.items():
    calib_k = calib_idx[archetype_test[calib_idx] == k]
    diagnostic_rows.append(
        {
            "Archetype": k,
            "Calibration rows": len(calib_k),
            "tau": round(tau_k, 4),
            "Mean sigma": round(float(scale[calib_k].mean()), 4),
        }
    )
cp_all = ConformalResidualFitting(alpha=0.1)
scores_all = cp_all.get_scores(loc[calib_idx], scale[calib_idx], y_test_flat[calib_idx])
tau_all = float(np.mean(cp_all.calculate_conformal_quantile(scores_all, alpha=0.1)))
diagnostic_rows.append(
    {
        "Archetype": "Global",
        "Calibration rows": len(calib_idx),
        "tau": round(tau_all, 4),
        "Mean sigma": round(float(scale[calib_idx].mean()), 4),
    }
)
diagnostic_df = pd.DataFrame(diagnostic_rows)
themed_gt(
    GT(diagnostic_df)
    .tab_header(title=md("**The Model's Own Sigma Barely Varies by Archetype; Tau Does**"))
    .tab_source_note("Mean sigma identical to 4 decimal places across every archetype"),
    n_rows=len(diagnostic_df),
)

GT(_tbl_data=  Archetype  Calibration rows     tau  Mean sigma
0         0              2868  0.7683      0.6154
1         1              4656  0.9759      0.6154
2         2              3368  0.8170      0.6154
3         3              2255  0.7885      0.6154
4         4              2243  0.9771      0.6154
5    Global             15390  0.8175      0.6154, _body=<great_tables._gt_data.Body object at 0x12f6934d0>, _boxhead=Boxhead([ColInfo(var='Archetype', type=<ColInfoTypeEnum.default: 1>, column_label='Archetype', column_align='left', column_width=None), ColInfo(var='Calibration rows', type=<ColInfoTypeEnum.default: 1>, column_label='Calibration rows', column_align='right', column_width=None), ColInfo(var='tau', type=<ColInfoTypeEnum.default: 1>, column_label='tau', column_align='right', column_width=None), ColInfo(var='Mean sigma', type=<ColInfoTypeEnum.default: 1>, column_label='Mean sigma', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12f6933b0>, _spanners=Spanners([]), _heading=Heading(title=Md(text="**The Model's Own Sigma Barely Varies by Archetype; Tau Does**"), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12f6938c0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12f6938f0>, _source_notes=['Mean sigma identical to 4 decimal places across every archetype'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Archetype', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Calibration rows', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='tau', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Mean sigma', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='Archetype', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='Archetype', rownum=3, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='Archetype', rownum=5, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='Calib

MLPGAMNormal's own predicted $\sigma$ is essentially constant across
every archetype, the model has not learned much real heteroscedasticity by
customer type. Real per-archetype differences do show up, but in $\tau$, not
$\sigma$: $\tau$ ranges from roughly 0.77 to 1.00 across archetypes against a
global 0.82. Calibration only needs the *tail* of a calibration set, the top
10% of scores, to estimate $\tau$ at $\alpha=0.1$; splitting an already-modest
calibration set five ways shrinks that effective tail sample per archetype,
adding real quantile-estimation noise that outweighs whatever conditioning
benefit archetype membership might otherwise offer. The archetypes were
built for shape similarity in Chapter 4, not for residual-scale
heterogeneity, and here, checked directly, they do not transfer to this
different question.

## One honest comparison, not one winner

Three paradigms have each been checked on their own terms above: a
parametric density that prices risk directly but assumes a Normal shape, a
quantile forecast that assumes nothing about shape but over-covered by
nearly nine points before calibration, and a split-conformal wrapper that
delivers an actual guarantee, sharper globally than per archetype. Putting
all four real numbers, the parametric MAE, the quantile paradigm's own
coverage and width, and both conformal variants, into one table makes the
full tradeoff visible at once, rather than scattered across four separate
results a reader has to hold in their head.

In [ ]:
summary_rows = pd.DataFrame(
    [
        {"Paradigm": "Parametric (Normal)", "MAE": round(float(normal_mae), 4), "Coverage": None, "Mean width": None},
        {
            "Paradigm": "Quantile (FPQR)",
            "MAE": None,
            "Coverage": round(float(fpqr_picp), 3),
            "Mean width": round(float(fpqr_width), 4),
        },
        {"Paradigm": "Conformal, global", "MAE": None, "Coverage": round(cov_g, 3), "Mean width": round(width_g, 4)},
        {
            "Paradigm": "Conformal, per-archetype",
            "MAE": None,
            "Coverage": round(cov_pa, 3),
            "Mean width": round(width_pa, 4),
        },
    ]
)
themed_gt(
    GT(summary_rows)
    .tab_header(title=md("**Four Real Numbers, Four Different Decisions**"))
    .tab_source_note("342 real AusNet customers, last-90-day holdout, MLPGAM backbone throughout"),
    n_rows=len(summary_rows),
)

GT(_tbl_data=                   Paradigm     MAE  Coverage  Mean width
0       Parametric (Normal)  0.3627       NaN         NaN
1           Quantile (FPQR)     NaN     0.990      3.0809
2         Conformal, global     NaN     0.899      1.0317
3  Conformal, per-archetype     NaN     0.899      1.1079, _body=<great_tables._gt_data.Body object at 0x12f6928a0>, _boxhead=Boxhead([ColInfo(var='Paradigm', type=<ColInfoTypeEnum.default: 1>, column_label='Paradigm', column_align='left', column_width=None), ColInfo(var='MAE', type=<ColInfoTypeEnum.default: 1>, column_label='MAE', column_align='right', column_width=None), ColInfo(var='Coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Coverage', column_align='right', column_width=None), ColInfo(var='Mean width', type=<ColInfoTypeEnum.default: 1>, column_label='Mean width', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12f6b5f70>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Four Real Numbers, Four Different Decisions**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12f6b5160>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12f6b5e50>, _source_notes=['342 real AusNet customers, last-90-day holdout, MLPGAM backbone throughout'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Paradigm', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Coverage', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Mean width', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font='Jost', size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Paradigm', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Paradigm', rownum=3, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='MAE', rownum=1, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='MAE', rownum=3, colnum=None, styles=[CellStyleFill(color='#EAF3FA')]), StyleInfo(locname=LocBody(columns=None, rows=

In [ ]:
width_plot_df = pd.DataFrame(
    [
        {"Approach": "Quantile (FPQR), raw", "width": float(fpqr_width)},
        {"Approach": "Conformal, global", "width": width_g},
        {"Approach": "Conformal, per-archetype", "width": width_pa},
    ]
)
(
    ggplot(width_plot_df, aes(x="Approach", y="width", fill="Approach"))
    + geom_bar(stat="identity", width=0.6)
    + scale_fill_manual(
        values={
            "Quantile (FPQR), raw": PARADIGM_COLORS["Quantile (FPQR)"],
            "Conformal, global": PARADIGM_COLORS["Conformal, global"],
            "Conformal, per-archetype": PARADIGM_COLORS["Conformal, per-archetype"],
        }
    )
    + labs(x="", y="Mean interval width (90% nominal)", title="Global Conformal Is the Sharpest Calibrated Interval")
    + modern_theme(legend_pos="none", x_axis_angle=12)
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

## Why bother

None of the three paradigms is strictly better; each answers a different
real question. The parametric head is the only one that prices a
risk-of-exceedance question directly. The quantile head is the only one
free of a distributional assumption, at the cost of needing its own
calibration check, its raw intervals over-covered by nearly nine points
here. Conformal calibration is the only one with an actual finite-sample
guarantee, and checking whether Chapter 2's own archetypes sharpen it
further was a real test, not an assumption: they do not, and the mechanism
behind that answer, a near-constant learned scale and a tail-sample
efficiency loss, is as real and useful a finding as a positive result would
have been.

## Where this leaves Part 5

A point forecast alone was never the full deliverable this part set out to
build. Chapter 2 found the model that earns its cost; this chapter attached
an honest number to how wrong that model's own forecasts might be, and
checked, rather than assumed, which calibration actually earns its own
added complexity.

## References

::: {#refs}
:::